# 








































Diagramas HR y R-masa con isocronas, por TIC

Para cada estrella de `catalogo_completo_mist_sommers.csv`, dos paneles:

- **HR** (Teff vs L): la curva de la isocrona de MIST a `age_median_1` y de
  Somers17 a `age_median_2`, con el punto observado de la **primaria**
  (Teff, L del ajuste de cada modelo) encima.
- **R vs masa**: las mismas isocronas, con la primaria en
  `(mass_median, R1)` (R1 vía Stefan-Boltzmann).

Cuando el TIC tiene además una estimación de la **secundaria**
(`teff2_desde_profundidad_transito.csv`, de `2_teff_desde_isocronas.ipynb`
-- R2 desde la profundidad del eclipse + `Teff(R,t)`), se agrega también:
su punto en ambos diagramas, y la curva de BHAC15 a la edad de MIST/Somers
(reutilizada porque BHAC15 no tiene su propio ajuste Teff/L, pero sí nos
interesa su malla para la secundaria).

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA_ORION = Path("/home/luciam/Documents/starry/Resultados_Orion")
RUTA_MASSAGE = RUTA_ORION / "datos_estrellas_massage_orion"
sys.path.insert(0, str(RUTA_MASSAGE))

from isocronas_interp import (
    load_isochrone_model,
    IsochroneInterpolator,
    invert_radius_to_mass,
    teff_de_radio_edad,
)

T_SUN = 5772.0  # K

## 1) Catálogos

- `cat`: Teff/L/masa/edad de la primaria por TIC (MIST `_1`, Somers17 `_2`).
- `sec`: R2/Teff2 de la secundaria por TIC+Sector (de `2_teff_desde_isocronas.ipynb`);
  puede no existir todavía si ese notebook no se ha corrido -- en ese caso
  simplemente no se dibuja el punto de la secundaria.

In [ ]:
cat = pd.read_csv(RUTA_ORION / "catalogo_completo_mist_sommers.csv")

sec_path = RUTA_MASSAGE / "teff2_desde_profundidad_transito.csv"
sec = pd.read_csv(sec_path) if sec_path.exists() else None

print(f"cat: {cat.shape}")
print(f"sec: {sec.shape if sec is not None else 'no encontrado -- se omiten los puntos de la secundaria'}")

## 2) Isocronas

In [ ]:
modelos_dir = RUTA_MASSAGE / "modelos"

grid_mist = load_isochrone_model(
    "MIST", modelos_dir / "iso_mist" / "MIST_iso_6a9a687177c08_todas_edades.csv"
)
grid_somers = load_isochrone_model(
    "Somers17", modelos_dir / "iso_somers17" / "f017track_ms.csv"
)
grid_bhac15 = load_isochrone_model(
    "BHAC15", modelos_dir / "iso_barafe" / "BHAC15_iso.txt"
)

interp_mist = IsochroneInterpolator(grid_mist, "MIST")
interp_somers = IsochroneInterpolator(grid_somers, "Somers17")
interp_bhac15 = IsochroneInterpolator(grid_bhac15, "BHAC15")

## 3) Funciones auxiliares

`curva_isocrona`: a una edad fija, barre la masa y regresa
`(masa, Teff, L, R)` para dibujar la isocrona completa en ambos diagramas
de una sola vez. `radio_stefan_boltzmann`/`lum_stefan_boltzmann` van y
vuelven entre (Teff, L) y R, igual que en `mass_age2.ipynb` /
`2_teff_desde_isocronas.ipynb`.

In [ ]:
def curva_isocrona(interp, edad_myr, n=300):
    if pd.isna(edad_myr):
        return None
    logAge = np.log10(edad_myr * 1e6)
    masa = np.linspace(interp.mass_min, interp.mass_max, n)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        teff = np.atleast_1d(interp.teff(logAge, masa))
        logL = np.atleast_1d(interp.logL(logAge, masa))
        radio = np.atleast_1d(interp.radius(logAge, masa))
    return masa, teff, 10.0 ** logL, radio


def radio_stefan_boltzmann(Teff, logL, T_sun=T_SUN):
    return 10 ** (logL / 2) * (T_sun / Teff) ** 2


def lum_stefan_boltzmann(Teff, R, T_sun=T_SUN):
    """L/Lsun a partir de Teff [K] y R [Rsun] (inversa de radio_stefan_boltzmann)."""
    return (R ** 2) * (Teff / T_sun) ** 4


for nombre, T_col, L_col in [("MIST", "T_median_1", "lgL_median_1"), ("Somers17", "T_median_2", "lgL_median_2")]:
    cat[f"R1_{nombre}"] = radio_stefan_boltzmann(cat[T_col], cat[L_col])

## 4) Diagramas por TIC

Curvas: MIST (azul, edad `age_median_1`), Somers17 (naranja, edad
`age_median_2`), BHAC15 reutilizando esas mismas edades (verde, punteado).
Puntos: primaria (círculo relleno, un color por modelo), secundaria si
existe (triángulo = modelo propio, diamante = BHAC15 con el R2/edad de ese
modelo). Los TIC sin ninguna edad disponible se omiten (no hay isocrona
que dibujar).

In [ ]:
output_dir = RUTA_ORION / "hr_diagrams"
output_dir.mkdir(parents=True, exist_ok=True)

for _, row in cat.iterrows():
    tic = int(row["TIC"])
    edad_mist, edad_somers = row["age_median_1"], row["age_median_2"]

    if pd.isna(edad_mist) and pd.isna(edad_somers):
        print(f"TIC {tic}: sin edad disponible en ningún modelo, se omite.")
        continue

    fig, (ax_hr, ax_rm) = plt.subplots(1, 2, figsize=(13, 5.5))

    # limites de los ejes se fijan por los PUNTOS observados, no por la
    # curva completa (que llega a masas de cientos de Msun y taparia la
    # zona de interes, de baja masa, en un rincon del panel)
    puntos_hr_T, puntos_hr_L = [], []
    puntos_rm_M, puntos_rm_R = [], []

    # -------- curvas de isocrona --------
    curvas = [
        ("MIST", edad_mist, interp_mist, "tab:blue", "-"),
        ("Somers17", edad_somers, interp_somers, "tab:orange", "-"),
        ("BHAC15 (edad MIST)", edad_mist, interp_bhac15, "tab:green", "--"),
        ("BHAC15 (edad Somers17)", edad_somers, interp_bhac15, "tab:green", ":"),
    ]
    for etiqueta, edad, interp, color, ls in curvas:
        curva = curva_isocrona(interp, edad)
        if curva is None:
            continue
        masa, teff, L, R = curva
        ax_hr.plot(teff, L, color=color, ls=ls, label=f"{etiqueta}, {edad:.3g} Myr")
        ax_rm.plot(masa, R, color=color, ls=ls, label=f"{etiqueta}, {edad:.3g} Myr")

    # -------- primaria observada, por modelo --------
    primarias = [
        ("MIST", "T_median_1", "lgL_median_1", "mass_median_1", "R1_MIST", "tab:blue"),
        ("Somers17", "T_median_2", "lgL_median_2", "mass_median_2", "R1_Somers17", "tab:orange"),
    ]
    for nombre, T_col, L_col, M_col, R1_col, color in primarias:
        if pd.isna(row[T_col]) or pd.isna(row[L_col]):
            continue
        T1, L1, M1, R1 = row[T_col], 10 ** row[L_col], row[M_col], row[R1_col]
        ax_hr.scatter(T1, L1, color=color, edgecolor="k", zorder=5, s=90,
                      label=f"primaria ({nombre})")
        puntos_hr_T.append(T1)
        puntos_hr_L.append(L1)
        if not pd.isna(M1):
            ax_rm.scatter(M1, R1, color=color, edgecolor="k", zorder=5, s=90,
                          label=f"primaria ({nombre})")
            puntos_rm_M.append(M1)
            puntos_rm_R.append(R1)

    # -------- secundaria, si existe --------
    if sec is not None:
        filas_sec = sec.loc[sec["TIC"] == tic]
        for _, s in filas_sec.iterrows():
            sector = s["Sector"]
            fuentes_sec = [
                ("MIST", "R2_MIST", "age_median_1", "Teff2_MIST", interp_mist, "tab:blue", "^"),
                ("Somers17", "R2_Somers17", "age_median_2", "Teff2_Somers17", interp_somers, "tab:orange", "^"),
                ("BHAC15/MIST", "R2_MIST", "age_median_1", "Teff2_BHAC15_desde_MIST", interp_bhac15, "tab:green", "D"),
                ("BHAC15/Somers17", "R2_Somers17", "age_median_2", "Teff2_BHAC15_desde_Somers17", interp_bhac15, "tab:green", "D"),
            ]
            for etiqueta, R2_col, age_col, Teff2_col, interp, color, marker in fuentes_sec:
                R2, edad, Teff2 = s[R2_col], s[age_col], s[Teff2_col]
                if pd.isna(R2) or pd.isna(edad) or pd.isna(Teff2):
                    continue
                L2 = lum_stefan_boltzmann(Teff2, R2)
                logAge = np.log10(edad * 1e6)
                mass2 = invert_radius_to_mass(logAge, R2, interp)
                ax_hr.scatter(Teff2, L2, color=color, marker=marker, edgecolor="k", zorder=5, s=70,
                              label=f"secundaria S{sector} ({etiqueta})")
                puntos_hr_T.append(Teff2)
                puntos_hr_L.append(L2)
                if not np.isnan(mass2):
                    ax_rm.scatter(mass2, R2, color=color, marker=marker, edgecolor="k", zorder=5, s=70,
                                  label=f"secundaria S{sector} ({etiqueta})")
                    puntos_rm_M.append(mass2)
                    puntos_rm_R.append(R2)

    # -------- estilo --------
    PAD = 4.0  # factor multiplicativo de margen alrededor de los puntos observados

    ax_hr.set_xscale("log")
    ax_hr.set_yscale("log")
    if puntos_hr_T:
        ax_hr.set_xlim(max(puntos_hr_T) * PAD, min(puntos_hr_T) / PAD)  # invertido: Teff alta a la izquierda
        ax_hr.set_ylim(min(puntos_hr_L) / PAD, max(puntos_hr_L) * PAD)
    else:
        ax_hr.invert_xaxis()
    ax_hr.set_xlabel("Teff [K]")
    ax_hr.set_ylabel("L [Lsun]")
    ax_hr.set_title(f"TIC {tic} -- diagrama HR")
    ax_hr.grid(alpha=0.25, linestyle="--")

    ax_rm.set_xscale("log")
    ax_rm.set_yscale("log")
    if puntos_rm_M:
        ax_rm.set_xlim(min(puntos_rm_M) / PAD, max(puntos_rm_M) * PAD)
        ax_rm.set_ylim(min(puntos_rm_R) / PAD, max(puntos_rm_R) * PAD)
    ax_rm.set_xlabel("Masa [Msun]")
    ax_rm.set_ylabel("R [Rsun]")
    ax_rm.set_title(f"TIC {tic} -- R vs masa")
    ax_rm.grid(alpha=0.25, linestyle="--")

    handles_hr, labels_hr = ax_hr.get_legend_handles_labels()
    by_label = dict(zip(labels_hr, handles_hr))  # sin duplicados (misma línea en ambos paneles)
    fig.legend(by_label.values(), by_label.keys(), loc="lower center",
               ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.08))

    fig.tight_layout(rect=[0, 0.05, 1, 1])
    fig.savefig(output_dir / f"TIC_{tic}_hr_rm.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 5) Diagrama HR por sector (estilo "divulgación")

Un solo panel por sector, con **todas** las estrellas (primarias + secundarias)
observadas en ese sector, inspirado en el diagrama HR clásico: color por
temperatura (azul = caliente, rojo/naranja = frío), tamaño del punto por
radio (con leyenda de radios solares de referencia), y etiqueta de TIC junto
a cada punto.

Para cada sistema se usa el primer modelo disponible en `modelo_orden`
(por default: MIST y si no, Somers17) -- así cada estrella aparece una sola
vez por diagrama, aunque tenga ajuste en ambos modelos.

**Nota:** las etiquetas de zonas espectrales ("Secuencia principal",
"Gigantes", etc., como en la imagen de referencia) están apagadas por
default (`mostrar_etiquetas_espectrales=False`) porque sus posiciones en el
ejemplo son manuales y fueron pensadas para un rango Teff/L que va de
supergigantes a enanas marrones -- muy distinto del rango que van a cubrir
tus binarias jóvenes. Si las activas, seguramente vas a necesitar mover las
posiciones (fx, fy) para que caigan donde sí hay estrellas.

In [ ]:
# -------- helpers: reunir todas las estrellas (primaria + secundaria) de un sector --------
# incluye masa (necesaria para el diagrama logTe-logL coloreado por masa de la sección 6).
# Para la SECUNDARIA, tanto Teff como la masa se toman de/contra BHAC15
# (Baraffe), no del interpolador del propio modelo (MIST/Somers17):
#   - Teff2_MIST/Teff2_Somers17 (Teff(R2, edad) contra la malla propia de
#     cada modelo) da NaN para la enorme mayoría de las secundarias, porque
#     son de muy baja masa y caen fuera de la malla de masa de esos modelos
#     (MIST/Somers17 empiezan en ~0.1 Msun) -- por eso casi ninguna
#     secundaria se dibujaba antes.
#   - BHAC15 sí cubre hasta 0.01 Msun, así que se usa Teff2_BHAC15_desde_MIST
#     / Teff2_BHAC15_desde_Somers17 (ya calculado en
#     `2_teff_desde_isocronas.ipynb`, reutilizando el R2/edad de cada rama)
#     como Teff de la secundaria, y la misma malla de BHAC15 para invertir
#     su masa -- igual que ya se hacía (bien) en la sección 4.
# La masa puede seguir saliendo NaN si ni así cae dentro de la malla -- en
# ese caso la estrella se conserva igual, solo que sin masa asignada.

SUFIJO_MODELO = {"MIST": "1", "Somers17": "2"}
AGE_COL_MODELO = {"MIST": "age_median_1", "Somers17": "age_median_2"}


def _valor_primaria(fila_cat, modelo):
    suf = SUFIJO_MODELO[modelo]
    T_col, L_col, R_col, M_col = (
        f"T_median_{suf}", f"lgL_median_{suf}", f"R1_{modelo}", f"mass_median_{suf}",
    )
    if pd.isna(fila_cat[T_col]) or pd.isna(fila_cat[L_col]):
        return None
    M = fila_cat[M_col] if M_col in fila_cat and not pd.isna(fila_cat[M_col]) else np.nan
    return fila_cat[T_col], 10 ** fila_cat[L_col], fila_cat[R_col], M


def _valor_secundaria(fila_sec, modelo):
    # "modelo" aquí selecciona de qué RAMA sale R2/edad (MIST o Somers17);
    # la Teff2 siempre se lee de la malla de BHAC15 evaluada en ese (R2, edad)
    # -- ver la explicación en el comentario de arriba.
    R_col, age_col = f"R2_{modelo}", AGE_COL_MODELO[modelo]
    Teff_col = f"Teff2_BHAC15_desde_{modelo}"
    if any(c not in fila_sec for c in (R_col, Teff_col, age_col)):
        return None
    R, Teff, edad = fila_sec[R_col], fila_sec[Teff_col], fila_sec[age_col]
    if pd.isna(R) or pd.isna(Teff) or pd.isna(edad):
        return None
    logAge = np.log10(edad * 1e6)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        M = invert_radius_to_mass(logAge, R, interp_bhac15)  # BHAC15: mejor ajuste para masas bajas
    return Teff, lum_stefan_boltzmann(Teff, R), R, M  # M puede ser NaN, se conserva la estrella


def estrellas_del_sector(sector, modelo_orden=("MIST", "Somers17")):
    """DataFrame con una fila por estrella (primaria 'A' y secundaria 'B')
    observada en `sector`, usando el primer modelo de `modelo_orden` que
    tenga Teff, L y R (la masa, columna `M`, puede quedar NaN)."""
    if sec is None:
        raise ValueError("No hay tabla de secundarias (`sec`) cargada.")

    filas_sector = sec.loc[sec["Sector"] == sector]
    if filas_sector.empty:
        return pd.DataFrame(columns=["tic", "comp", "modelo", "Teff", "L", "R", "M"])

    estrellas = []
    for tic in filas_sector["TIC"].unique():
        tic = int(tic)

        # -------- primaria (una vez por TIC, viene de `cat`) --------
        fila_cat = cat.loc[cat["TIC"] == tic]
        if not fila_cat.empty:
            fila_cat = fila_cat.iloc[0]
            for modelo in modelo_orden:
                val = _valor_primaria(fila_cat, modelo)
                if val is not None:
                    Teff, L, R, M = val
                    estrellas.append(dict(tic=tic, comp="A", modelo=modelo, Teff=Teff, L=L, R=R, M=M))
                    break

        # -------- secundaria(s) en ese sector --------
        for _, fila_sec in filas_sector.loc[filas_sector["TIC"] == tic].iterrows():
            for modelo in modelo_orden:
                val = _valor_secundaria(fila_sec, modelo)
                if val is not None:
                    Teff, L, R, M = val
                    estrellas.append(dict(tic=tic, comp="B", modelo=modelo, Teff=Teff, L=L, R=R, M=M))
                    break

    return pd.DataFrame(estrellas)


In [ ]:
# -------- función de graficado, inspirada en el diagrama HR de referencia --------

def plot_hr_sector(sector, modelo_orden=("MIST", "Somers17"),
                    mostrar_nombres=True, mostrar_etiquetas_espectrales=False,
                    etiquetas_espectrales=None):
    df = estrellas_del_sector(sector, modelo_orden=modelo_orden)
    if df.empty:
        print(f"Sector {sector}: no hay estrellas con datos suficientes, se omite.")
        return

    fig, ax = plt.subplots(figsize=(9, 7))

    # -------- color por temperatura: azul (caliente) -> rojo/naranja (frío) --------
    cmap = plt.get_cmap("RdYlBu_r")
    norm = plt.Normalize(vmin=df["Teff"].min(), vmax=df["Teff"].max())
    colores = cmap(1 - norm(df["Teff"]))  # invertido: Teff alta -> extremo "frío" del cmap (azul)

    # -------- tamaño por radio, en escala log para que quepan enanas y gigantes --------
    def tamano(R):
        return 25 + 55 * np.log10(np.clip(R, 1e-2, None) / 1e-2)

    ax.scatter(df["Teff"], df["L"], s=tamano(df["R"]), c=colores,
               edgecolor="black", linewidth=0.4, alpha=0.9, zorder=5)

    if mostrar_nombres:
        for _, e in df.iterrows():
            ax.annotate(f"TIC {e['tic']}{e['comp']}", (e["Teff"], e["L"]),
                        textcoords="offset points", xytext=(5, 4),
                        fontsize=6.5, alpha=0.85)

    # -------- leyenda de tamaños (radios solares de referencia, como en el ejemplo) --------
    radios_ref = [r for r in [10, 1, 0.1, 0.01]
                  if df["R"].min() / 3 <= r <= df["R"].max() * 3] or [1]
    handles = [plt.scatter([], [], s=tamano(r), color="0.6", edgecolor="black",
                            linewidth=0.4, label=f"{r:g}") for r in radios_ref]
    leg = ax.legend(handles=handles, title="Radio\n(radios solares)", loc="lower left",
                     fontsize=8, title_fontsize=8, labelspacing=1.3, frameon=False)
    ax.add_artist(leg)

    # -------- etiquetas de regiones espectrales (opcional, posiciones manuales) --------
    if mostrar_etiquetas_espectrales:
        etiquetas_espectrales = etiquetas_espectrales or [
            ("Secuencia principal", 0.5, 0.45, -35),
            ("Gigantes", 0.3, 0.75, 0),
        ]
        for texto, fx, fy, rot in etiquetas_espectrales:
            ax.text(fx, fy, texto, transform=ax.transAxes, fontsize=10,
                    style="italic", rotation=rot, ha="center", color="0.3")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.invert_xaxis()  # Teff alta a la izquierda, como en el diagrama HR clásico
    ax.set_xlabel("Temperatura efectiva (K)")
    ax.set_ylabel("Luminosidad (luminosidades solares)")
    ax.set_title(f"Diagrama Hertzsprung-Russell -- Sector {sector}")
    ax.grid(alpha=0.2, linestyle="--")

    fig.tight_layout()
    fig.savefig(output_dir / f"HR_sector_{sector}.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


SECTORES = sorted(sec["Sector"].unique())

for sector in SECTORES:
    plot_hr_sector(sector)


## 6) Diagrama log Te vs log L, color = masa, tamaño = radio

Mismo conjunto de estrellas por sector que en la sección 5
(`estrellas_del_sector`), pero ahora:

- **eje X**: log Teff, **eje Y**: log (L/Lsun) -- ejes lineales sobre los
  valores ya logaritmizados (en vez de ejes log-log sobre Teff/L directos).
- **color**: masa (escala log de color, `viridis`), con barra de color.
- **tamaño**: radio (misma idea que antes: proporcional a log R, con
  leyenda de radios de referencia).
- **forma del marcador**: `*` para primarias, `o` para secundarias --
  así se distinguen aunque compartan color/tamaño.

In [ ]:
# -------- diagrama logTe-logL: color por masa, tamaño por radio, forma por A/B --------
from matplotlib.colors import LogNorm

def plot_hr_sector_masa(sector, modelo_orden=("MIST", "Somers17"), mostrar_nombres=True):
    df = estrellas_del_sector(sector, modelo_orden=modelo_orden)
    if df.empty:
        print(f"Sector {sector}: no hay estrellas con datos suficientes, se omite.")
        return

    # las que sí tienen masa válida definen la escala de color; las que no
    # (comunmente secundarias fuera de la malla de masa del modelo) se
    # dibujan igual, pero en gris, para no perderlas del diagrama
    con_masa = df.dropna(subset=["M"])
    sin_masa = df.loc[df["M"].isna()]
    if con_masa.empty:
        print(f"Sector {sector}: ninguna estrella con masa válida -- se grafican todas en gris.")

    fig, ax = plt.subplots(figsize=(9, 7))

    cmap = plt.get_cmap("viridis")
    if not con_masa.empty:
        norm = LogNorm(vmin=con_masa["M"].min(), vmax=con_masa["M"].max())

    # -------- tamaño por radio, en escala log (mismo criterio que la sección 5) --------
    def tamano(R):
        return 25 + 55 * np.log10(np.clip(R, 1e-2, None) / 1e-2)

    # las estrellas ('*') se ven visualmente más chicas que un círculo de la
    # misma "s", así que se les da un pequeño extra de área para emparejar
    marcadores = {"A": ("*", 1.8), "B": ("o", 1.0)}

    for comp, (marker, factor) in marcadores.items():
        # -------- con masa: coloreadas según el colormap --------
        sub = con_masa.loc[con_masa["comp"] == comp]
        if not sub.empty:
            ax.scatter(np.log10(sub["Teff"]), np.log10(sub["L"]),
                       s=tamano(sub["R"]) * factor, c=sub["M"], cmap=cmap, norm=norm,
                       marker=marker, edgecolor="black", linewidth=0.4,
                       alpha=0.9, zorder=5)
        # -------- sin masa: mismo marcador y tamaño, pero en gris --------
        sub_gris = sin_masa.loc[sin_masa["comp"] == comp]
        if not sub_gris.empty:
            ax.scatter(np.log10(sub_gris["Teff"]), np.log10(sub_gris["L"]),
                       s=tamano(sub_gris["R"]) * factor, color="0.75",
                       marker=marker, edgecolor="black", linewidth=0.4,
                       alpha=0.9, zorder=4)

    if mostrar_nombres:
        for _, e in df.iterrows():
            ax.annotate(f"TIC {e['tic']}{e['comp']}",
                        (np.log10(e["Teff"]), np.log10(e["L"])),
                        textcoords="offset points", xytext=(5, 4),
                        fontsize=6.5, alpha=0.85)

    # -------- barra de color: masa, con el rango explícito en la etiqueta --------
    if not con_masa.empty:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        cbar.set_label(f"Masa (M$_\odot$)  [{con_masa['M'].min():.2f} - {con_masa['M'].max():.2f}]")
        # ticks parejos en escala log entre el mínimo y el máximo real de este sector
        ticks_masa = np.geomspace(con_masa["M"].min(), con_masa["M"].max(), 5)
        cbar.set_ticks(ticks_masa)
        cbar.set_ticklabels([f"{t:.2g}" for t in ticks_masa])

    # -------- leyenda de tamaños (radios solares de referencia) --------
    radios_ref = [r for r in [10, 1, 0.1, 0.01]
                  if df["R"].min() / 3 <= r <= df["R"].max() * 3] or [1]
    handles_r = [plt.scatter([], [], s=tamano(r), color="0.6", edgecolor="black",
                              linewidth=0.4, label=f"{r:g}") for r in radios_ref]
    leg_r = ax.legend(handles=handles_r, title="Radio\n(radios solares)", loc="lower left",
                       fontsize=8, title_fontsize=8, labelspacing=1.3, frameon=False)
    ax.add_artist(leg_r)

    # -------- leyenda de forma: primaria vs secundaria (+ "sin masa" si aplica) --------
    handles_ab = [
        plt.scatter([], [], marker="*", s=140, color="0.6", edgecolor="black", label="Primaria"),
        plt.scatter([], [], marker="o", s=80, color="0.6", edgecolor="black", label="Secundaria"),
    ]
    if not sin_masa.empty:
        handles_ab.append(
            plt.scatter([], [], marker="s", s=80, color="0.75", edgecolor="black",
                        label="Sin masa estimada")
        )
    ax.legend(handles=handles_ab, loc="upper right", fontsize=8, frameon=False)

    ax.invert_xaxis()  # log Teff alto a la izquierda, como en el diagrama HR clásico
    ax.set_xlabel("log Teff [K]")
    ax.set_ylabel("log (L / L$_\odot$)")
    ax.set_title(f"Diagrama HR (log Te vs log L) -- Sector {sector}")
    ax.grid(alpha=0.2, linestyle="--")

    fig.tight_layout()
    fig.savefig(output_dir / f"HR_logTL_masa_sector_{sector}.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


SECTORES = sorted(sec["Sector"].unique())

for sector in SECTORES:
    plot_hr_sector_masa(sector)


## 7) Tabla resumen por TIC

Una fila por **(TIC, Sector)** -- la primaria es la misma para todos los
sectores de un TIC, pero la secundaria cambia porque R2 sale de la
profundidad del eclipse medida en cada sector. Reutiliza exactamente los
mismos valores que ya se grafican en las secciones 5 y 6
(`_valor_primaria` / `_valor_secundaria` / `SECTORES`): mismo modelo
elegido por `modelo_orden`, misma Teff2 de secundaria vía BHAC15, misma
masa invertida contra BHAC15.

Columnas:

- `..._A`: primaria (Teff, L, R, masa, edad y qué modelo -- MIST o
  Somers17 -- se usó, según `modelo_orden`).
- `..._B`: secundaria "elegida" para los diagramas (primera rama
  disponible en `modelo_orden`) -- idéntica a una de las dos columnas
  siguientes, con su incertidumbre en `Teff_B_err` / `M_B_err`.
- `..._B_BHAC15_MIST` / `..._B_BHAC15_Somers17`: el ajuste de la
  secundaria contra la malla de **Baraffe (BHAC15)**, por separado para
  cada rama de (R2, edad) -- la de MIST y la de Somers17 -- sin importar
  cuál se haya elegido arriba. Es el mismo Teff2(R2, edad) + inversión de
  masa que ya se usaba en la sección 4/5, ahora expuesto explícitamente
  para las dos ramas a la vez.

**Incertidumbre de la secundaria** (columnas `_err`, sección "Incertidumbre
de la secundaria" más abajo): se propagan dos fuentes de error hasta
Teff2 y M2, por diferencias finitas sobre la malla de BHAC15
(`incertidumbre_secundaria`):

- `Depth_primary_err` (error de la profundidad del tránsito, en
  `priors_completosv1.csv` / `sec`) -> `R_..._err` (sigma de R2, vía
  R2 = R1·sqrt(Depth) con R1 fijo).
- `age_std_1` / `age_std_2` (error de la edad, en
  `catalogo_completo_mist_sommers.csv`) -> `edad_..._err_Myr`.

No se propaga el propio error de Teff/L de la primaria (R1), que no se
pidió incluir. Si alguna combinación cae fuera de la malla de BHAC15 en
ambos extremos (±sigma), la incertidumbre queda en NaN en vez de un
número engañosamente pequeño.

Se guarda completa en `resumen_diagramas_hr_por_tic.csv`.

In [ ]:
# -------- incertidumbre de la secundaria: Teff2 y M2 --------
# Dos fuentes de error, tal como vienen en los catálogos del proyecto:
#   - sigma_R2, de Depth_primary_err (profundidad del tránsito, en `sec` /
#     `priors_completosv1.csv`), vía R2 = R1 * sqrt(Depth) con R1 fijo
#     (no se propaga aquí el propio error de T/L de la primaria, que no se
#     pidió incluir): sigma_R2 / R2 = 0.5 * sigma_Depth / Depth.
#   - sigma_edad, de age_std_1 / age_std_2 (en `cat`, `catalogo_completo_mist_sommers.csv`).
# Ambas se propagan a Teff2 y M2 por diferencias finitas alrededor de
# (R2, edad) sobre la malla de BHAC15 (la misma que ya da Teff2 y M2).

AGE_ERR_COL_MODELO = {"MIST": "age_std_1", "Somers17": "age_std_2"}


def _semi_diff(hi, lo, mid):
    """(hi-lo)/2 si ambos lados son finitos; si solo uno lo es, la
    diferencia de un solo lado contra el valor central; NaN si ninguno
    (p. ej. R2+sigma_R2 y R2-sigma_R2 caen los dos fuera de la malla)."""
    hi_ok, lo_ok, mid_ok = np.isfinite(hi), np.isfinite(lo), np.isfinite(mid)
    if hi_ok and lo_ok:
        return (hi - lo) / 2
    if hi_ok and mid_ok:
        return hi - mid
    if lo_ok and mid_ok:
        return mid - lo
    return np.nan


def incertidumbre_secundaria(R2, sigma_R2, edad_Myr, sigma_edad_Myr, interp):
    """sigma_Teff2, sigma_M2 propagadas desde sigma_R2 y sigma_edad_Myr
    contra la malla `interp` (BHAC15), por diferencias finitas centradas en
    (R2, edad_Myr). NaN si R2/edad_Myr no son válidos, o si la malla no
    puede evaluarse ni siquiera de un lado."""
    if pd.isna(R2) or pd.isna(edad_Myr):
        return np.nan, np.nan

    logAge = np.log10(edad_Myr * 1e6)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        Teff0 = teff_de_radio_edad(R2, edad_Myr, interp)
        M0 = invert_radius_to_mass(logAge, R2, interp)

        dTeff_R, dM_R = 0.0, 0.0
        if pd.notna(sigma_R2) and sigma_R2 > 0:
            R_hi, R_lo = R2 + sigma_R2, max(R2 - sigma_R2, 1e-4)
            Teff_hi, Teff_lo = teff_de_radio_edad(R_hi, edad_Myr, interp), teff_de_radio_edad(R_lo, edad_Myr, interp)
            M_hi, M_lo = invert_radius_to_mass(logAge, R_hi, interp), invert_radius_to_mass(logAge, R_lo, interp)
            dTeff_R = _semi_diff(Teff_hi, Teff_lo, Teff0)
            dM_R = _semi_diff(M_hi, M_lo, M0)

        dTeff_age, dM_age = 0.0, 0.0
        if pd.notna(sigma_edad_Myr) and sigma_edad_Myr > 0:
            edad_min_myr, edad_max_myr = 10 ** interp.logAge_min / 1e6, 10 ** interp.logAge_max / 1e6
            edad_hi = min(edad_Myr + sigma_edad_Myr, edad_max_myr)
            edad_lo = max(edad_Myr - sigma_edad_Myr, edad_min_myr)
            Teff_hi, Teff_lo = teff_de_radio_edad(R2, edad_hi, interp), teff_de_radio_edad(R2, edad_lo, interp)
            M_hi = invert_radius_to_mass(np.log10(edad_hi * 1e6), R2, interp)
            M_lo = invert_radius_to_mass(np.log10(edad_lo * 1e6), R2, interp)
            dTeff_age = _semi_diff(Teff_hi, Teff_lo, Teff0)
            dM_age = _semi_diff(M_hi, M_lo, M0)

    sigma_Teff = np.nan if np.isnan(dTeff_R) or np.isnan(dTeff_age) else float(np.hypot(dTeff_R, dTeff_age))
    sigma_M = np.nan if np.isnan(dM_R) or np.isnan(dM_age) else float(np.hypot(dM_R, dM_age))
    return sigma_Teff, sigma_M

In [ ]:
def tabla_resumen(modelo_orden=("MIST", "Somers17")):
    filas = []
    for sector in SECTORES:
        filas_sector = sec.loc[sec["Sector"] == sector]
        for tic in filas_sector["TIC"].unique():
            tic = int(tic)
            fila = {"TIC": tic, "Sector": sector}

            fila_cat_series = None
            filas_cat_match = cat.loc[cat["TIC"] == tic]
            if not filas_cat_match.empty:
                fila_cat_series = filas_cat_match.iloc[0]
                for modelo in modelo_orden:
                    val = _valor_primaria(fila_cat_series, modelo)
                    if val is not None:
                        Teff, L, R, M = val
                        fila.update({
                            "modelo_A": modelo,
                            "edad_A_Myr": fila_cat_series[AGE_COL_MODELO[modelo]],
                            "Teff_A": Teff, "L_A": L, "R_A": R, "M_A": M,
                        })
                        break

            for _, fila_sec in filas_sector.loc[filas_sector["TIC"] == tic].iterrows():
                # -------- ajuste de Baraffe (BHAC15) de la secundaria, con
                # su incertidumbre, una columna por cada rama de (R2, edad)
                # -- MIST y Somers17 -- sin importar cuál se use luego como
                # "la" secundaria. sigma_Teff2/sigma_M2 vía
                # `incertidumbre_secundaria` (sigma_R2 de Depth_primary_err,
                # sigma_edad de age_std_1/2). --------
                for modelo in ("MIST", "Somers17"):
                    val = _valor_secundaria(fila_sec, modelo)
                    sufijo = f"B_BHAC15_{modelo}"
                    if val is not None:
                        Teff, L, R, M = val
                        edad_myr = fila_sec[AGE_COL_MODELO[modelo]]

                        Depth = fila_sec.get("Depth_primary", np.nan)
                        Depth_err = fila_sec.get("Depth_primary_err", np.nan)
                        sigma_R2 = (
                            R * 0.5 * abs(Depth_err / Depth)
                            if pd.notna(Depth) and pd.notna(Depth_err) and Depth != 0
                            else np.nan
                        )

                        sigma_edad_myr = np.nan
                        if fila_cat_series is not None:
                            col_err = AGE_ERR_COL_MODELO[modelo]
                            if col_err in fila_cat_series and pd.notna(fila_cat_series[col_err]):
                                sigma_edad_myr = fila_cat_series[col_err]

                        sigma_Teff, sigma_M = incertidumbre_secundaria(
                            R, sigma_R2, edad_myr, sigma_edad_myr, interp_bhac15,
                        )

                        fila.update({
                            f"edad_{sufijo}_Myr": edad_myr,
                            f"Teff_{sufijo}": Teff, f"L_{sufijo}": L,
                            f"R_{sufijo}": R, f"M_{sufijo}": M,
                            f"R_{sufijo}_err": sigma_R2,
                            f"edad_{sufijo}_err_Myr": sigma_edad_myr,
                            f"Teff_{sufijo}_err": sigma_Teff,
                            f"M_{sufijo}_err": sigma_M,
                        })
                    else:
                        fila.update({
                            f"edad_{sufijo}_Myr": np.nan,
                            f"Teff_{sufijo}": np.nan, f"L_{sufijo}": np.nan,
                            f"R_{sufijo}": np.nan, f"M_{sufijo}": np.nan,
                            f"R_{sufijo}_err": np.nan,
                            f"edad_{sufijo}_err_Myr": np.nan,
                            f"Teff_{sufijo}_err": np.nan,
                            f"M_{sufijo}_err": np.nan,
                        })

                # -------- secundaria "elegida" para los diagramas de las
                # secciones 5/6: la primera rama disponible en modelo_orden,
                # con su incertidumbre copiada de la columna BHAC15_{modelo}
                # correspondiente (ya calculada arriba) --------
                for modelo in modelo_orden:
                    val = _valor_secundaria(fila_sec, modelo)
                    if val is not None:
                        Teff, L, R, M = val
                        sufijo = f"B_BHAC15_{modelo}"
                        fila.update({
                            "modelo_B": modelo,
                            "edad_B_Myr": fila[f"edad_{sufijo}_Myr"],
                            "Teff_B": Teff, "L_B": L, "R_B": R, "M_B": M,
                            "Teff_B_err": fila[f"Teff_{sufijo}_err"],
                            "M_B_err": fila[f"M_{sufijo}_err"],
                        })
                        break

            filas.append(fila)

    resumen = pd.DataFrame(filas)
    if "OBJECT" in cat.columns:
        resumen = resumen.merge(cat[["TIC", "OBJECT", "RA", "DEC"]], on="TIC", how="left")

    cols_orden = [
        "TIC", "OBJECT", "Sector", "RA", "DEC",
        "modelo_A", "edad_A_Myr", "Teff_A", "L_A", "R_A", "M_A",
        "modelo_B", "edad_B_Myr", "Teff_B", "L_B", "R_B", "M_B",
        "Teff_B_err", "M_B_err",
        "edad_B_BHAC15_MIST_Myr", "edad_B_BHAC15_MIST_err_Myr",
        "Teff_B_BHAC15_MIST", "Teff_B_BHAC15_MIST_err",
        "L_B_BHAC15_MIST", "R_B_BHAC15_MIST", "R_B_BHAC15_MIST_err",
        "M_B_BHAC15_MIST", "M_B_BHAC15_MIST_err",
        "edad_B_BHAC15_Somers17_Myr", "edad_B_BHAC15_Somers17_err_Myr",
        "Teff_B_BHAC15_Somers17", "Teff_B_BHAC15_Somers17_err",
        "L_B_BHAC15_Somers17", "R_B_BHAC15_Somers17", "R_B_BHAC15_Somers17_err",
        "M_B_BHAC15_Somers17", "M_B_BHAC15_Somers17_err",
    ]
    cols_orden = [c for c in cols_orden if c in resumen.columns]
    resto = [c for c in resumen.columns if c not in cols_orden]
    return resumen[cols_orden + resto].sort_values(["TIC", "Sector"]).reset_index(drop=True)


resumen = tabla_resumen()

csv_path = RUTA_ORION / "resumen_diagramas_hr_por_tic.csv"
resumen.to_csv(csv_path, index=False)
print(f"Guardado: {csv_path}  ({len(resumen)} filas, {resumen['TIC'].nunique()} TICs)")

resumen